In [19]:
import kagglehub
import pandas as pd
import numpy as np
import os

try:
    path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
    data = pd.read_csv(os.path.join(path, 'Monday-WorkingHours.pcap_ISCX.csv'))
    print(" Path to dataset files:", path)
    
except Exception as e:
    print(f"Something failed... {e}")

data.head()

 Path to dataset files: /Users/antoniogonzalez/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data.columns = data.columns.str.strip()

data['Label'] = le.fit_transform(data['Label'])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

X = data.drop('Label', axis=1)
y = data['Label']

In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

#  Use scaler to normalize all features to the same scale.
# Prevent larger values from dominating smaller values preventing the model
# from becoming biased to bigger values.
scaler = StandardScaler()

#  fit_transform calculates the scale from X and applies it.
#  Returning everything as a numpy array,
#  we do this so that something like 1.00 and 0.01 dont get mixed up
#  and clash with each other
X_scaled = scaler.fit_transform(X) 

#  Keeping the train/test split consistently 80/20% as all the rest of the models
#  To keep an acurate comparison. 
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=100)

#  Utlizing the data to torch tensors to our study and exam
#  This allows tensor to memorize their results and improve
#  The model as the model is on run time.
#  FloatTensor = decimal numbers (features)
#  LongTensor = whole numbers (labels/classes)
X_train_t = torch.FloatTensor(X_train) # Studying for practice exam
X_test_t = torch.FloatTensor(X_test)  #  Studying for exam
y_train_t = torch.LongTensor(y_train.values) #  Taking practice exam
y_test_t = torch.LongTensor(y_test.values) #  Taking the actual exam


#  Utlizing the Neural netowrk of torch
#  Using torch module to run the model itself
class NeuralNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NeuralNet, self).__init__()
        self.layers = nn.Sequential(
            #  Layer 1: compress 79 features down to 128 neurons
            nn.Linear(input_size, 128), # Linear satistical model of TORCH
            nn.ReLU(), # Ignores negative signals, pass positive ones.
            nn.Linear(128, 64), # Layer 2: compress 128 neurons down to 64
            nn.ReLU(), # Same as before
            nn.Linear(64, num_classes) # Output layer one score per class
        )
    def forward(self, x):
        return self.layers(x) #  Pass data through all layers in order
    
#  The number of input features which is (79 columns)
input_size = X_train_t.shape[1]

#  The actual number of unique attack classes within the dataset
num_classes = len(y.unique()) 

#  The model with input and output sizes assigned to 'model'
model = NeuralNet(input_size, num_classes)

#  CrossEntrophyloss: measures how wrong the model's predictions 
#  actually were.
#  (lower loss = better prediction)
criterion = nn.CrossEntropyLoss()

# Were adjusting the model's weight after each batch
# lr = 0.001 controls how big each adjustment step is
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#  Finally taking 'X_train_t' and 'y_train_t' 
#  actually wrapping it into a dataset object for Pytorch to then use.
dataset = TensorDataset(X_train_t, y_train_t)

#  DATA LORDER - Feeds data to the model in batches of 512 rows at a time
#  shuffle=True - ensures each epoch sees the data to ensure its refeeding 
# new imformaiton preventing overfitting the model
loader = DataLoader(dataset, batch_size=512, shuffle=True)


#  iterating to a range of 10 epochs which one full pass through all training data
# is one epoch 
for epoch in range(10):
    for X_batch, y_batch in loader:
        optimizer.zero_grad()               #  Reset gradients from previous batch
        output = model(X_batch)             #  Foward pass: makes the actual prediction             
        loss = criterion(output, y_batch)   #  Calculate how wrong the model was
        loss.backward()                     #  Backwards pass: figure out what needs to be fixed
        optimizer.step()                    #  Update weights based on what the model learned
    print(f'Epoch {epoch+1}/10 Loss: {loss.item():.4f}')

    #  Evaluate: Turns off training mode so model stops updating the weight
    model.eval()
    with torch.no_grad():  #  No more track gradients during evaluation
         y_pred = model(X_test_t).argmax(dim=1).numpy()  #  Always choosing the highest score class.

    print(classification_report(y_test, y_pred))

Epoch 1/10 Loss: 0.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105897

    accuracy                           1.00    105897
   macro avg       1.00      1.00      1.00    105897
weighted avg       1.00      1.00      1.00    105897

Epoch 2/10 Loss: 0.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105897

    accuracy                           1.00    105897
   macro avg       1.00      1.00      1.00    105897
weighted avg       1.00      1.00      1.00    105897

Epoch 3/10 Loss: 0.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105897

    accuracy                           1.00    105897
   macro avg       1.00      1.00      1.00    105897
weighted avg       1.00      1.00      1.00    105897

Epoch 4/10 Loss: 0.0000
              precision    recall  f1-score   support

           0       1.00      1